# Second Year Paper Code

#### Author: Mariana Oseguera
Date: June 27th, 2022


##  1. Import Packages


In [169]:
import pandas as pd
import numpy as np
import csv
import gc

In [69]:
import string 
import nltk

In [4]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords as sw

In [5]:
from nltk.collocations import *
bg = nltk.collocations.BigramAssocMeasures()

In [6]:
from collections import Counter

In [7]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

In [8]:
import random
from multiprocessing import  Pool

In [286]:
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import statsmodels.api as sm
from pathlib import Path

In [9]:
from pathlib import Path

## 2. Import datasets

### 2.1 Compustat Table

In [9]:
### Import Crosswalk 
df_compustat = pd.read_stata('/Users/mgor/Documents/Strategy/2YP/Data/compustat_bg_names_dedupl.dta')

### Rename employer
df_compustat=df_compustat.rename(columns={"name_bgt" :"Employer"})

### 2.2 Main BG Table

In [10]:
### Drop main that are not in compustat and append a year into one dataframe 
### Declare Path
path_main=Path('/Users/mgor/Documents/Strategy/2YP/Data/data_raw/Main')
df_main=pd.DataFrame()

### Select employers that are report on compustat

for file in path_main.glob('*.txt'):
    print(file.name)
    df_aux= pd.read_table(file, encoding='latin')
    df_aux=df_aux.merge(df_compustat, how='left', on='Employer')
    df_aux=df_aux.dropna(subset=['gvkey'])
    df_main=df_main.append(df_aux)

Main_2022-01.txt


/opt/anaconda3/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3063: DtypeWarning: Columns (28,29,30) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


Main_2022-02.txt


In [11]:
## Save merged main

df_main.to_csv('/Users/mgor/Documents/Strategy/2YP/Data/data_raw/Main/main_2022.csv',index=False)

### 2.3 Job Text Table 

In [12]:
### Merge with job text with main
df_2022=pd.DataFrame()

### Declare path
path_txt=Path('/Users/mgor/Documents/Strategy/2YP/Data/data_processed/2022')

for file in path_txt.glob('*.csv'):
    print(file.name)
    df_aux= pd.read_table(file, delimiter=',', header=0)
    df_aux=df_aux.rename(columns={"JobID" :"BGTJobId"})
    df_aux=df_aux.merge(df_main, how='inner', on='BGTJobId')
    df_2022=df_2022.append(df_aux)

US_XML_AddFeed_20220120_20220126.csv
US_XML_AddFeed_20220217_20220223.csv
US_XML_AddFeed_20220106_20220112.csv
US_XML_AddFeed_20220203_20220209.csv
US_XML_AddFeed_20220303_20220309.csv
US_XML_AddFeed_20220224_20220302.csv
US_XML_AddFeed_20220127_20220202.csv
US_XML_AddFeed_20220310_20220316.csv
US_XML_AddFeed_20220210_20220216.csv
US_XML_AddFeed_20211230_20220105.csv
US_XML_AddFeed_20220113_20220119.csv


In [330]:
#df_2022.to_csv('/Users/mgor/Documents/Strategy/2YP/Data/data_processed/2022/merged_2022.csv', index=False)


## 3. Dictionaries 

### 3.1 Wilmers Zhang Paper

In [10]:
### Read file 
df_wilzha=pd.read_table('/Users/mgor/Documents/Strategy/2YP/Data/data_csv/key_words.csv', delimiter=',', header=0)


In [11]:
### Select list of words Wilmers and Zhang
full_list=df_wilzha.full_list.str.strip().str.replace('_', ' ').dropna().tolist()
career_meaning=df_wilzha.career_meaning.str.strip().str.replace('_', ' ').dropna().tolist()
worklife_meaning=df_wilzha.worklife_meaning.str.strip().str.replace('_', ' ').dropna().tolist()

### 3.2 NLP for SDGs from Amel-Zadeh, Amir and Chen, Mike and Mussalli, George and Weinberg, Michael, NLP for SDGs: Measuring Corporate Alignment with the Sustainable Development Goals (June 26, 2021). Columbia Business School Research Paper , Available at SSRN: https://ssrn.com/abstract=3874442 or http://dx.doi.org/10.2139/ssrn.3874442

 (a) From Table 2 from the paper which contains the final seed words
 (b) The second process substract repetitive words 
 (c) Plus ESG Match Table 3
 (d) Plus words in the UN SDG goal definition

In [12]:
###1.No Poverty
SDG1=['social', 'end poverty', 'poverty dimension', 'poverty', 'social protection', 'poor', 'unemployed person', 'poverty line', 'protection',
'cash benefit', 'extreme poverty', 'poor and vulnerable', 'humanitarian', 'vulnerable']

remove=['social','end poverty', 'extreme poverty','poverty dimension', 'poverty line', 'poor', 'protection','poor vunerable'] ## (b)
SDG1 = list(set(SDG1) - set(remove))
SDG1.append('community')## (c)
#SDG1.append('poorest people')## (d)


###2. Zero Hunger
SDG2=['malnutrition', 'hunger', 'food producer', 'underweight', 'hunger and malnutrition', 'undernutrition', 'famine', 'food insecurity', 'agricultural productivity',
      'agricultural', 'extreme hunger', 'agriculture', 'prevalence of undernourishment', 'nutritional need', 'food']

remove=['food producer','agricultural', 'agriculture', 'food','agricultural productivity' ] ## (b)
SDG2 = list(set(SDG2) - set(remove))
# SDG2.extend(['food security', 'sustainable agriculture', 'improved nutrition', 'food prices','adequate food', 'food shortages'])## (d)


###3. Good Health and Well-Being
SDG3=['life expectancy', 'mental health', 'air pollution', 'medicine vaccine', 'infectious disease', 'good health', 'respiratory disease',
'reproductive health', 'mortality, health care', 'healthcare','disease diabetes', 'disease', 'health coverage', 'health', 'maternal mortality', 'death preventable',
 'cardiovascular disease', 'infectious disease']
#SDG3.extend(['healthy lives','well being ','wellbeing','global health',  'cardiovascular disease',
            #'death preventable']) ## (d)


###4. Quality of Education
SDG4= ['teacher', 'secondary school', 'proficiency level', 'primary school', 'inclusive', 'literacy', 'literacy numeracy', 'higher education',
'quality education', 'school', 'effective learn', 'vocational training', 'level proficiency', 'minimum proficiency', 'technical vocational']

remove=(['teacher','secondary school', 'proficiency level', 'primary school','literacy', 'literacy numeracy','higher education','quality education', 'school', 'effective learn',
      'level proficiency', 'minimum proficiency', 'technical vocational', 'vocational training','inclusive'] ) ## (b)
SDG4 = list(set(SDG4) - set(remove))
SDG4.extend(['people development', 'worker development', 'training', 'career development', 'skill development', 
         'skills development', 'disabilities','inclusive development']) ## (c)
#SDG4.extend(['learning opportunities']) ## (d)


###5. Gender Equality
SDG5=['domestic work', 'right', 'sexual violence', 'women', 'girl', 'discrimination', 'reproductive health', 'managerial position', 'woman girl',
      'marriage', 'woman representation', 'gender equality', 'gender parity', 'child marriage', 'gender']

remove=['right','managerial position', 'woman girl','gender equality', 'gender parity', 'child marriage',
           'woman representation', 'marriage'] ##(b)
SDG5 = list(set(SDG5) - set(remove))
SDG5.extend(['diversity and inclusion','diversity and opportunity','equal opportunity','equal opportunities', 
        'female representation']) ## (c)
#SDG5.extend(['equality']) ## (d)


###6. Clean Water and Sanitation
SDG6= ['water and sanitation', 'drink water', 'sanitation', 'basic drink', 'waste water', 'water scarcity', 'hygiene', 'water', 'sanitation service',
'sanitation hygiene', 'supply of freshwater', 'handwash facility', 'resource management', 'water stress']
remove=['hygiene', 'water', 'sanitation service','sanitation hygiene',
           'handwash facility', 'sanitation', 'basic drink'] ##(b)
SDG6 = list(set(SDG6) - set(remove))
SDG6.extend(['water efficiency','toxic chemicals reduction', 'waste reduction', 'waste recycling', 'biodiversity impact'])## (c)
#SDG6.extend(['sustainable management', 'water stress', 'water related ecosystems']) ## (d)


###7. Affordable and Clean Energy
SDG7=['energy', 'technology', 'renewable energy', 'infrastructure', 'electricity', 'cheap energy', 'solar energy', 'solar power', 'windpower',
      'thermal power', 'energy productivity', 'energy efficiency', 'greenhouse gases', 'greenhouse', 'fossil fuels', 'pollution', 'energy standards',
      'energy access','energy consumption', 'access to electricity', 'without electricity', 'fuel technology','fossil fuel']
remove=['energy', 'technology','infrastructure', 'electricity', 'greenhouse']
SDG7 = list(set(SDG7) - set(remove))## (b)
SDG7.extend(['responsible use']) ### (c)
#SDG7.extend(['clean energy','environmental sustainability', 'sustainable energy']) ##(d)


###8. Decent Workand Economic Growth
SDG8=['labor', 'employment', 'gdp', 'job', 'unemployed', 'economic growth', 'productivity', 'job creation', 'slavery', 'forced labor',
      'labor force', 'women participation', 'labor organization','human right', 'informal employment', 
      'growth rate', 'labour productivity','decent work','secure work','global economic','gender pay','crisis level', 'rate real',
      'education employment','human slavery', 'child labour', 'youth employment']
remove=['labor','employment', 'gdp', 'job', 'unemployed', 'economic growth', 'productivity',
       'labor force', 'growth rate', 'labour productivity', 'rate real',
       'education employment','slavery human']
SDG8 = list(set(SDG8) - set(remove))## (b)
SDG8.extend(['child labor', 'human rights' ])
#SDG8.extend(['employee development','professional development','professional growth','decent employment', 'decent conditions','decent job', 'complete benefits','full benefits' ])

###9. Industry, Innovation and Infrastructure
SDG9= ['research development', 'development', 'industry', 'infrastructure', 'transport', 'technological progress', 'communication technology',
       'sustainable development', 'sustainable industries', 'innovation', 'entrepreneurship', 'access information','internet access', 'material footprint',
       'developed country', 'least developed', 'economic infrastructure','infrastructure support','global manufacture',
       'scientific research', 'resilient infrastructure', 'research and innovation']
remove=['development', 'industry', 'infrastructure', 'transport', 'communication technology', 'economic infrastructure','infrastructure support',
        'global manufacture','innovation', 'entrepreneurship', ]
SDG9 = list(set(SDG9) - set(remove))## (b)

#SDG9.extend(['sustainable industrialization', 'foster innovation'])##(d)
SDG9.extend(['environmental innovation'])##(c)

###10. Reduced Inequalities 
SDG10=['income', 'population', 'income inequality', 'economic inclusion', 'safe migration', 'economic inequality', 
       'reduce inequality', 'wealth share', 'indigenous rights', 'migrant workers', 'migrant', 'official development', 
       'income inequality','migration mobility','global wealth', 'least develop' , 'development assistance']
remove=['income', 'population', 'population convenient','official development' , 'least develop']
SDG10 = list(set(SDG10) - set(remove))## (b)
#SDG10.extend(['racial equality','equal opportunity','inequality reduction','reduction of inequality', 'economic opportunity','inequality of opportunity','economic mobility'])##(d)

###11. Sustainable Cities and Communities
SDG11=['city', 'urban', 'urban population','public', 'disability', 'disaster', 'sustainable city', 'affordable housing', 'housing access',
       'resilient societies', 'public transport', 'public spaces', 'urban planning', 'inclusive', 'business opportunities',
       'sustainable development', 'person with disability', 'green public', 'sustainable resilient','sustainable urbanization','population convenient', 
       'convenient access','migrant',]
remove=['city', 'urban','public','public transport', 'public spaces','inclusive', 'urban planning', 'business opportunities',
       'population convenient', 'person with disability']
SDG11 = list(set(SDG11) - set(remove))## (b)
SDG11.extend(['slums','sustainable cities'])

###12. Responsible Consumption and Production
SDG12=['responsible consumption', 'sustainable development', 'resources', 'consumption', 'production', 'development',
'reduce waste', 'efficient', 'efficient economy', 'energy consumption', 'energy efficient', 'supporting developing',
'material footprint', 'natural resource','recycle','sustainable consumption','domestic material','consumption production','food waste']
remove=['resources', 'consumption', 'production', 'development','efficient', 'natural resource','consumption production']
SDG12 = list(set(SDG12) - set(remove))## (b)
SDG12.extend(['environmental materials sourcing','sustainable packaging', 'environmental supply chain', 'responsible supply chain management','recycling initiatives','resource use','sustainability reporting', 'fuel efficiency']) ##(c)

###13.  Climate Action
SDG13=['climate', 'develop', 'disaster', 'local', 'emissions reductions', 'global warming','climate change', 'climate system',
'greenhouse gas', 'greenhouse gas emissions', 'co2 emissions', 'low carbon', 'disaster risk', 'sustainable management', 'natural resource management', 'sea levels',
       'sustainable energy','paris agreement',  'climate relate','green energy']
remove=['climate','develop', 'disaster', 'local']
SDG13 = list(set(SDG13) - set(remove))## (b)
SDG13.extend(['carbon emissions','fossil fuels emissions', 'wind power','reduce emissions'])##(c)
#SDG13.extend(['environmental action','rising temperatures','environmental activist','environmental activism','environmental consequences'])


###14. Life Below Water
SDG14=['marine', 'ocean', 'fish', 'sea', 'water', 'fishery', 'overfishing', 'coastal biodiversity', 'coastal ecosystems',
       'ocean resources', 'marine biodiversity', 'fish stocks', 'marine pollution', 'ocean acidification', 'depleted fisheries',
       'unregulated fish','fish stock', 'fishery management','marine coastal','fishery subsidy','conservation',
       'marine technology']
remove=['marine', 'ocean', 'fish', 'sea', 'water', 'fishery','marine coastal','fishery subsidy']
SDG14 = list(set(SDG14) - set(remove))## (b)
#SDG14.extend(['ocean warming','plastic pollution'])##(d)

###15. Life on Land
SDG15=['land degradation', 'terrestrial freshwater', 'ecosystem', 'deforestation', 'specie animal', 'forest management', 'biodiversity',
'conservation', 'protect area', 'forest, terrestrial', 'species', 'wildlife', 'protect', 'agriculture', 'land', 'area']
remove=['ecosystem','conservation','specie animal','forest, terrestrial','species','protect', 'agriculture', 'land', 'area']
SDG15 = list(set(SDG15) - set(remove)) ## (b)

## SDG15.extend(['environmental ecosystem','environmental conservation', 'animal species','terrestial ecosystem','combat desertification','environmental project', 'wild life'])
 


### 16. Peace, Justice, and Strong Institutions
SDG16=['law', 'human right', 'rights violation', 'insecurity', 'institution', 'violence', 'exploitation', 'global governance',
'transparency', 'rule', 'corruption and bribery', 'access to justice', 'corruption', 'justice', 'peace', 'conflict and violence', 'international',
'victim human', 'conflict']
remove=['corruption' ,'institution','rule','corruption bribery','international','victim human', 'conflict']
SDG16 = list(set(SDG16) - set(remove))## (b)
SDG16.extend(['peaceful','inclusive society'])

###17. Partnerships for the Goals
SDG17=['sustainable development', 'innovation', 'enhance international', 'official development', 'capacity building', 'development',
'coordinate policy', 'develop country', 'assistance', 'cooperation', 'international support', 'international cooperation',
'partnership', 'international', 'policy coherence', 'development assistance', 'country','official development','development',
      ]
remove=['innovation', 'enhance international', 'develop country', 'assistance',
       'partnership', 'international', 'policy coherence',  'country','official development','development']
SDG17 = list(set(SDG17) - set(remove))## (b)


In [13]:
### Create list of the all the SDGs dictionaries 
SDG_dict=SDG1+SDG2+SDG3+SDG4+SDG5+SDG6+SDG7+SDG8+SDG9+SDG10+SDG11+SDG12+SDG13+SDG13+SDG14+SDG15+SDG16+SDG17

In [14]:
### Create list with unique elements
unique_list = []
for x in SDG_dict:
        # check if exists in unique_list or not
        if x not in unique_list:
            unique_list.append(x)
SDG_dict=unique_list   

### 3.3 Pencle & Mălăescu (2016)– Corporate Social Responsibility

In [544]:
### Read file 
df_pema=pd.read_table('/Users/mgor/Documents/Strategy/2YP/Dictionaries/Pencle N & Malaescu/Keywords by subdivision.csv', delimiter=',', header=0)


In [545]:
### Create list of words
employee=df_pema.Employee.str.strip().str.replace('-', ' ').dropna().tolist()
environment=df_pema.Environment.str.strip().str.replace('-', ' ').dropna().tolist()
hum_rig=df_pema['Human Rights'].str.strip().str.replace('-', ' ').dropna().tolist()
social_com=df_pema['Social Community'].str.strip().str.replace('-', ' ').dropna().tolist()



In [546]:
### Convert all to lower case and remove punctuation

for i in range(len(employee)):
    employee[i] = employee[i].lower()
    employee[i] = employee[i].replace('[^\w\s]', '')

for i in range(len(environment)):
    environment[i] = environment[i].lower()
    environment[i] = environment[i].replace('[^\w\s]', '')
    
for i in range(len(hum_rig)):
    hum_rig[i] = hum_rig[i].lower()
    hum_rig[i] = hum_rig[i].replace('[^\w\s]', '')
    
for i in range(len(social_com)):
    social_com[i] = social_com[i].lower()
    social_com[i] = social_com[i].replace('[^\w\s]', '')




In [547]:
### Remove words to avoid double counting or words that dont apply to job posts:
### Example 1: Remove pollution as there is alredy: plastic pollution
### Example 2: Remove the word job 

remove=['alaska natives','environment', 'environmental','enhancing','environmental protection agency','fair hiring practices','alaskan natives','americans with disabilities act','benefit','charitable foundation','charitable giving','abuse','attention','accommodation','accountability','adopted child','adopted children','alcohol','balancing','beneficially','beneficiary','body','bonus','bylaws','certification','certifications','certify','certifying','claims','contribution','custodian','customs','development','died','dies','accept','accepted',
        'activities','adopt','adopted','adverse','adversely','affluence','affluences','agreements','agricultural','agriculture','agro','aids','air filtration','amazon','animal',
        'anti','arms','assurance','attributable','audit','auditor','auditors','authenticate','authenticity','awareness','barge','baselines','basin','beautiful',
        'director','diversification','diversified','diversifying','drug','educate','educating','education','educational','elected','employ','employed','employee','employees','employees','employer',
        'employers','employers','employing','employment','employs','beauty','board','boundaries','broad','bromides','bromine','building','bull','burn','cage','carbon','carbonate','carbonated','carbonates','carbons','carrying capacities','carrying capacity',
        'carrying load','carrying loads','catastrophic','chemicals','chloride','chlorine','city','civil','enabling','enhancements','enhancing ','engage','engaging','equal','ergonomically','ethnic','even','exercise','experience','experienced','fiduciary','families','family','goal','goals','governance','hard work',
        'climate','code','conflict mineral','conflict minerals','corn','counties','countries','country ','covenants','crops','crud','cultivation','cycle','delegation','demographic','design','dioxide','dioxides','disclosing','disclosure','disposal','diverse','health','hire','hiring','humans','incentives','individual','individually','infringe','infringement','infringing','insurance','involved ','jobs',
        'knowledge','knowledgeable','knowledgebase','labor','laborers','lawfulness','laws','leader','leaders','leadership','learned','legal','life benefits','management','mate','meals','diversify','dwindling','easements','efficiencies','efficiently','emissions','emission','medicinal','multinational','nonemployee','nonrenewal','occupational','officer','officers',
        'outsourcing','paid','participant','participants','participating','participatory','parties','partner','payroll','peer','people','performance','performers','person','personal','personnel','persons','philosophies','positions','practices','productivity','prescribed','privileges','environmental ','environmentally','epa standards','equip','evolution','exceed capacity','exceeded capacity',
        'exceeds capacity','cleanliness','country','employee well being',"employers'","employee's well being",'excess capacity','excess capacities','exit','expand','facility','farm fresh','farm','farmer','farmland','farmlands','flammability','flies','foodservice','fossil','fossils','professional','professionals','promotion','quality','rate','reallocate','reallocated','recognition',
        'recognize','recognized','regulate','regulations','regulatory ','reimburse','relations','relationship','relationships','religious ','respects','responsibility','responsible ','right ','role','safe','safety','salaries','seasonal','selection','sensitivity','served','serves','service','services','free range animals','free range','GRI',
        'fundraising','funds','genetically modified','gold','groves','grow','guidelines','harm','harmony','hazardous','hungry','hybrid','sick','size','spousal relationship','spousal relationships','spouse','strengths','suitability ','suitable','sustain','sustains','talented','team','teams','tenure','trained','truthfulness','understand','undocumented','unemployable','unproductive',
        'vision benefit','vision benefits','wage','wear','employee wellness','employee welfare','implementing','improve','improvements','medicare','medicaid','social wellbeing','indications','independent','KLD','land conservation','land conservationism','land conservationist','lives','living','locale','maintenance','maps','materials','maximum capacity','maximum capacities','members',
        'natural resource','natural','organic','overcapacity','ozone','petroleum','pipeline','free range animal','plant','pollution prevention','power','accompanied','accommodating','acts','african','africans','agent','aged','alternative lifestyle','alternative lifestyles','alternative life','avoid','award','awareness','belonging','certification','accountancy','affordable housing','american','bribe','coach',
        'committee','constitution','core','died','disadvantage','diversified','duty','civic','cleaned','clean','cleaner','cleaning','cleanliness ','cleanup','collective wellbeing','collective well being','collective wellbeings','collective well beings','common','community','concern','corporate foundation','county','educated','election','enabling ','engaged','ethnic','drinking','disclosures','disable','diet','eyes',
        'face','god given right','god given rights','hire','honest','interest','involuntarily','involuntary','jeopardizing','jeopardizes','jeopardize','involvement','involved','involve','intelligent','human','help','hope','governments','government','giving','future','fund','foodbank','foodbanks','food pantry','food pantries','labor','lawful','legality','medicinal','nationality','nationalization',
        'nationalized','native peoples','outsiders','outsourced','outsources','ownership','partnership','outperform','orphan','orphans','open','nature','naturally','native people','multinational','minimize','learning','lead','protector','rebuilding','regulation','regulatory','religious','right','publicly','projects','prevented','poor','plan','people group','people groups','party','owns','sexually',
        'social','strength','outsource','partnerships','vote','voting','workday','worker','workers','workforce','workforces','workplaces','workspaces','roll','responsible','respect','renovation','renew','rely','reliability','reduces','reduce','redeemable','recovery','socially','societal','sponsors','sponsorship','sustain','sustaining','sustains','trustees','united','unrestricted','uprooting','urban planning','urban','urbanization',
        'voluntarily','extended family','enhancing','bromines','voluntary','volunteer','volunteers','employees well being',"sponsors'",
        'risk tanking','sustainable','sustained','wheelchair access','vote','corporate','suitability',"employees'",'water', "workers'",
        'harness wind energy','mission', 'harness wind power','health benefits', 'health care benefits','healthy','interests',
        'entrepreneurial spirit','impact on local communities','inclusive development','inclusive society','waters','work','world','wrongdoers',
        'jeopardize','join a team','kld categories','partners','stakeholders','kld standards','land conservationists','meaningful ways','wrongdoing','wrongfully','zone',
        'zones', 'gri frameworks', 'gri ratings', 'gri standards' ]
add=['wellbeing','well being','core']

employee=list(set(employee) - set(remove)) 
environment=list(set(environment) - set(remove)) 
hum_rig=list(set(hum_rig) - set(remove)) 
social_com=list(set(social_com) - set(remove)) 

### Create list with unique elements


pema=employee+environment+hum_rig+social_com+add

### Create list with unique elements
unique_list = []
for x in pema:
        # check if exists in unique_list or not
        if x not in unique_list:
            unique_list.append(x)
pema=unique_list

### 3.4 Banks

In [20]:
### Read file 
#### Select dictionary's subsection: Development, Inclusiveness, CSR, Values-based, and Well-being
df_banks=pd.read_table('/Users/mgor/Documents/Strategy/2YP/Dictionaries/RegruitSig_Banks19/keywords_banks.csv', delimiter=',', header=0)
banks=df_banks.Development.str.strip().str.replace('[^\w\s]', '').dropna().tolist()+df_banks.Inclusiveness.str.strip().str.replace('[^\w\s]', '').dropna().tolist()+df_banks['Corporate Social Responsibility'].str.strip().str.replace('[^\w\s]', '').dropna().tolist()+df_banks['Values-based culture'].str.strip().str.replace('[^\w\s]', '').dropna().tolist()+df_banks['Well-Being'].str.strip().str.replace('[^\w\s]', '').dropna().tolist()

### Create list with unique elements
unique_list = []
for x in banks:
        # check if exists in unique_list or not
        if x not in unique_list:
            unique_list.append(x)
banks=unique_list

### Remove words to avoid double counting or words that dont apply to job posts:
remove=['ability', 'assignment','annual', 'assignments', 'develop', 'developed', 'developing', 'development', 'discover', 'discoveries', 'discovery', 'encourage', 'enhance', 'expand', 'explore','explores','experience', 'experienced', 'expertise', 'handson', 'leader', 'leadership', 'quality', 'role', 'staff', 'talent', 'talented', 'accommodation', 'countries', 'clean', 'efficiency', 'food', 'natural', 'plant', 'plants', 'power', 'exceptional', 'incentive', 'incentives', 'plan', 'service', 'skills', 'local', 'share', 'national', 'skill', 'sustain', 'goals', 'improvement', 'salary', 'highest', 'gift', 'energy', 'supply', 'advanced', 'able', 'live', 'pay', 'skilled', 'planning', 'personnel', 'recognize', 'suppliers', 'partner', 'fuel']

banks = list(set(banks) - set(remove)) 

### 3.5 Intersection Dictionary

In [1]:
### Remove words + Add words
#main_dict = list(set(own) - set(remove)) 
## Add from:
## https://www.mckinsey.com/business-functions/people-and-organizational-performance/our-insights/help-your-employees-find-purpose-or-watch-them-leave
## https://www.mckinsey.com/business-functions/people-and-organizational-performance/our-insights/the-search-for-purpose-at-work
## https://www.pwc.com/us/en/about-us/corporate-responsibility/assets/pwc-putting-purpose-to-work-purpose-survey-report.pdf
## https://www.greatplacetowork.com/resources/blog/purpose-at-work-predicts-if-employees-will-stay-or-quit-their-jobs
### https://www.limeade.com/resources/blog/help-employees-find-a-sense-of-purpose-in-the-workplace/

env=['greenhouse gas','wind energy', 'water usage','climate crisis','solar energy','sustainable growth', 'sustainable growth',
         'environmental footprint','environmental practices','responsible farming', 'green growth','sustainable practices',
        'water stress','water scarcity','water efficiency','toxic chemicals reduction','waste reduction','sustainable outcomes',
        'waste recycling', 'solar power','material footprint','fuel technology','wind power', 'thermal power', 'renewable energy', 'energy consumption',
        'responsible use','sustainable industries','sustainable development','responsible consumption', 'circular economy','sustainable packaging','environmetal supply chain','resource use',
        'fuel efficiency','global warming','paris agreement','green energy','fish stock','ocean resources','overfishing',
        'ocean acidification', 'planet through','sustainable urbanization','deforestation', 'local environment',
        'responsibility towards the environment','environmental action', 'environmental protection','climate neutral', 'carbon neutral', 
        'environmentally neutral', 'climate neutrality','carbon neutrality']
empl=['human element','secure work','worker development', 'interesting work','engaging work','challenging work','family time','meaning of work','great place to work','employee happiness','purpose at work', 'compassionate leadership',
         'organizational health','worker representation','workers voice','employees voice',
         'training programs','continuous learning','decent work','leave','career development','skills development','skill development',
         'recognized and rewarded','healthier and happier','accelerate the journey', 'health care']
humright=['peace','respect and dignity','justice', 'slavery','slums','forced labor','human right','access to electricity','energy access','fair hiring practices','disabilities','discrimination','sexual violence','girl','equal opportunities']
soc=['our mission', 'our purpose','responsible investment','align with the purpose','same values','purpose drives','societal value',
         'responsible business','values driven', 'purpose in business','purpose of the company', 'companys purpose',
         'organizations purpose','organizational purpose', 'wellbeing','collective action','for the community','hunger',
         'purpose driven','force for good','sense of purpose','real world','purpose of work', 'purpose of our work',
         'building a better world','work that matters','build a better world','role in society','impact on society',
          'impact on the world','social awareness','responsible management','poor and vulnerable','poverty','humanitarian',
        'social protection','vulnerable', 'food insecurity','malnutrition','famine','health coverage', 'peoples development',
        'inclusive development','technological progress','least developed','economic inequality','global wealth',
        'housing access', 'resilient societies','inclusive society', 'compassion','core values',
        'make a differece','compassionate','our values','making a difference','give back','positively impact',
        'charitable','peoples lives', 'charity','real difference','enriching','giving back','improve the lives', 'mission driven',
        'life changing','embodies','lives of millions', 'humanity', 'tends of thousands', 'live healthier', 'shape the future',
        'purposeful','core belief','improving the lives','lasting impact','meaningful ways','improve people',
        'positively affect','tens of millions','values driven','hundred of millions','someones life',
        'positively impacts','people in need','give back', 'transforming lives', 'donating',
        'transforming peoples', 'purpose of improving','core beliefs','touches millions','improves the lives',
        'tangible impact','donates', 'possitively affects','materially impact', 'improve mankind',
        'materially affect', 'improves or saves','life changing', 'lifechanging','renewing minds','stakeholder alignment',
        'differencemaker', 'missiondriven', 'purposedriven','strenghthenning communities', 'positive impact',
        'your impact','the lives of', 'impact on the environment','impact on people','impact on youth',
        'impact in the community','impact in someones life', 'impact on society','right thing','stakeholderoriented',
        'change the world','the live of', 'serving the community','benefit society','benefit the world', 'make the world better'
        'responsibility to the world', 'responsibility to society', 'income inequality', 'reduce inequality', 'inequality reduction',
        'inequality of opportunities', 'sdgs','stakeholder capitalism','equitable impact', 'rethink capitalism',
        'solving global problems','role of business','stakeholder oriented','stakeholdercentric','stakeholder centric',
        'long lasting impact','reimagine capitalism','bathrooms', 'class', 'reimagining capitalist', 'rethinking capitalism']

employee_final=employee+empl+worklife_meaning+career_meaning
environment_final=environment+env
hum_rig_final=hum_rig+humright
social_com_final=social_com+soc

employee_final=list(set(employee_final) - set(remove)) 
environment_final=list(set(environment_final) - set(remove)) 
hum_rig_final=list(set(hum_rig_final) - set(remove)) 
social_com_final=list(set(social_com_final) - set(remove)) 

NameError: name 'employee' is not defined

In [549]:
unique_list = []
for x in employee_final:
        # check if exists in unique_list or not
        if x not in unique_list:
            unique_list.append(x)
employee_final=unique_list


unique_list = []
for x in environment_final:
        # check if exists in unique_list or not
        if x not in unique_list:
            unique_list.append(x)
environment_final=unique_list

unique_list = []
for x in hum_rig_final:
        # check if exists in unique_list or not
        if x not in unique_list:
            unique_list.append(x)
hum_rig_final=unique_list

unique_list = []
for x in social_com_final:
        # check if exists in unique_list or not
        if x not in unique_list:
            unique_list.append(x)
social_com_final=unique_list

In [550]:
main_dict=employee_final+environment_final+hum_rig_final+social_com_final


In [553]:
len(main_dict)

623

In [552]:
unique_list = []
for x in main_dict:
        # check if exists in unique_list or not
        if x not in unique_list:
            unique_list.append(x)
main_dict=unique_list

### 3.6 File of all dictionaries

In [554]:
dictionaries=pd.DataFrame()

In [555]:
dictionaries = pd.DataFrame(
  {"main": pd.Series(main_dict), "pencle_malanescu":pd.Series(pema),"employee":pd.Series(employee_final),"environment":pd.Series(environment_final),"human_rights":pd.Series(hum_rig_final),"soc_com":pd.Series(social_com_final), "amel_etal":pd.Series(SDG_dict), "wilmers_zhang_main":pd.Series(full_list),"wilmers_zhang_career":pd.Series(career_meaning),"wilmers_zhang_life":pd.Series(worklife_meaning)})


In [556]:
dictionaries.to_csv('/Users/mgor/Documents/Strategy/2YP/Dictionaries/dictionaries.csv')

In [212]:
df_dict=pd.read_table('//Users/mgor/Documents/Strategy/2YP/Dictionaries/dictionaries.csv', delimiter=',', header=0)

full_list=df_dict.wilmers_zhang_main.str.strip().str.replace('_', ' ').dropna().tolist()
career_meaning=df_dict.wilmers_zhang_career.str.strip().str.replace('_', ' ').dropna().tolist()
worklife_meaning=df_dict.wilmers_zhang_life.str.strip().str.replace('_', ' ').dropna().tolist()
SDG_dict=df_dict.amel_etal.str.strip().str.replace('_', ' ').dropna().tolist()
pema=df_dict.pencle_malanescu.str.strip().str.replace('_', ' ').dropna().tolist()
banks=df_dict.banks.str.strip().str.replace('_', ' ').dropna().tolist()
main_dict=df_dict.main.str.strip().str.replace('_', ' ').dropna().tolist()


## 4. Job Text Cleaning and Word Search

In [28]:
df_2022=pd.read_table('/Users/mgor/Documents/Strategy/2YP/Data/data_processed/2022/merged_2022.csv.zip', delimiter=',', header=0)


/opt/anaconda3/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3063: DtypeWarning: Columns (34) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [218]:
### Paralellize function

def parallelize_dataframe(df, func, n_cores=2):
    df_split = np.array_split(df, n_cores)
    pool = Pool(n_cores)
    df = pd.concat(pool.map(func, df_split))
    pool.close()
    pool.join()
    return df

In [219]:
### Definition of count function   
def keyword_count(doc, keyword_list):
    return sum([doc.count(keyword) for keyword in keyword_list])

In [220]:
### Take sub sample of df_text
#df_sub= df_text.loc[1:10000] ###subselect sample of job text
df_sub=df_test.loc[1:100]

In [221]:
### Define stopwords and punctuation
stopwords=sw.words("english")
stopwords= stopwords+['apply', 'job','jobs', 'sales'] ##Remove other common words
punct = set(string.punctuation)

In [222]:
### No Punctuation
df_sub.JobText = df_sub.JobText.str.strip().str.replace('[^\w\s]', '')

In [234]:
def text_features(df):
    ### a. Make the original lower case and create a new column removing stopwords
    df['JobText'] = df['JobText'].apply(lambda x: ' '.join([item.lower() for item in x.split()]))
    df['JobText_pro'] = df['JobText'].apply(lambda x: ' '.join([item.lower() for item in x.split() if item not in stopwords]))
    df['tokenize_jobtext']=df['JobText_pro'].apply(nltk.word_tokenize)
    df['length'] = df.tokenize_jobtext.apply(lambda x: len(x))
    
    ### Creating list of frequencies 
    df['WordFreq']=df['tokenize_jobtext'].apply(nltk.FreqDist)
    
    ### Most Common Word
    df['MostCom'] = df.WordFreq.apply(lambda x: x.most_common(1))
    
    ### Separate Most Common Word into Word and Frequency + Relative Frequency 
    df['ComWord']=df['MostCom'].apply(lambda x: x[0][0])
    df['Freq']=df['MostCom'].apply(lambda x: x[0][1])
    df['Rel_Freq']=df['Freq']/df['length']
    
    ### Apply count function 
    
    ### Wilmers and Zhang Dictionary
    df['Full_wilzha']= df.JobText.apply(lambda x: keyword_count(x, full_list))/df['length']
    df['career_meaning']= df.JobText.apply(lambda x: keyword_count(x, career_meaning))/df['length']
    df['worklife_meaning']= df.JobText.apply(lambda x: keyword_count(x, worklife_meaning))/df['length']
    
    ###  Amel-Zadeh et al (2021) SDGs Dictionary
    df['SDGs']=df.JobText.apply(lambda x: keyword_count(x, SDG_dict))/df['length']
    
    ### Pencle & Mălăescu (2016)– Corporate Social Responsibility
    df['CSR']=df.JobText.apply(lambda x: keyword_count(x, pema))/df['length']
    
    ### Banks–Recruiting Signals of CSR, Development, Wellness, Inclusiveness
    df['Recru_sig']=df.JobText.apply(lambda x: keyword_count(x, banks))/df['length']
    
    ### Intersection
    df['main_dict']=df.JobText.apply(lambda x: keyword_count(x, main_dict))/df['length']
   

    return df

In [224]:
train = pd.DataFrame()
for i in range(16):
    print("Started processing: " + str(i*(100000)) + '. ' + 'Time is: ' + datetime.datetime.now().strftime("%H:%M:%S"))
    aux = parallelize_dataframe(df_2022.iloc[i*(100000):(100000)*(i+1)-1], text_features)
    train = train.append(aux)
    

NameError: name 'datetime' is not defined

In [235]:
aux = parallelize_dataframe(df_sub, text_features)


### 3.1 Extra Process for Purpose (Archive)

In [577]:
## Find words before and after purpose
df_sub=df_sub.assign(WordsBefore=df_sub.JobText.str.extract('(^\D+(?=purpose))'))
df_sub['WordsAfter']=df_sub.JobText.str.extract('((?<=purpose)\D+)')

In [578]:
### Subselect only the first and the last word
df_sub['WordsBefore'] = df_sub['WordsBefore'].fillna('')
df_sub['WordsAfter'] = df_sub['WordsAfter'].fillna('')

df_sub['WordsBefore'] = df_sub.WordsBefore.str.extract(r'((\b\w+)[\.?!\s]*$)')[0]
df_sub['WordsAfter']=df_sub['WordsAfter'].str.extract(r"(\w+)")[0]

df_sub['WordsBefore'] = df_sub['WordsBefore'].fillna('')
df_sub['WordsAfter'] = df_sub['WordsAfter'].fillna('')

## 4. Merge 

### 4.1 Merge main database with compustat, only for this year(rest in the server)

In [221]:
### Rename column 
df_compustat=df_compustat.rename(columns={"name_bgt" :"Employer"})


In [237]:
### Merge BGT Main and Compustat Key
df_comp=df_main.merge(df_compustat, how='left', on='Employer')


In [239]:
### Drop job posts of firms that are not in compustat
df_comp=test.dropna(subset=['gvkey'])

In [254]:
### Rename Job Id to Match Job Text Dataframe
train=train.rename(columns={"JobID" :"BGTJobId"})

In [271]:
### Merge compustat main with job text dataset
test=train.merge(df_comp, how='left', on='BGTJobId')

In [272]:
test=test.dropna(subset=['gvkey'])

In [276]:
train.JobDate

,Unnamed: 0,BGTJobId,JobReferenceID,JobDate,JobText,IsDuplicate,IsDuplicateOf,JobText_pro,tokenize_jobtext,length,...,ComWord,Freq,Rel_Freq,Full_wilzha,career_meaning,worklife_meaning,SDGs,CSR,Recru_sig,main_dict
1,1,39153020701,NaN,2021-12-30,apply to this job think youre the perfect cand...,False,NaN,think youre perfect candidate manufacturing co...,"[think, youre, perfect, candidate, manufacturi...",108,...,machines,6,0.055556,0.000000,0.000000,0.0,0.000000,0.055556,0.092593,0.074074
2,2,39153020721,NaN,2021-12-30,apply to this job think youre the perfect cand...,False,NaN,think youre perfect candidate looking career r...,"[think, youre, perfect, candidate, looking, ca...",865,...,tower,14,0.016185,0.001156,0.001156,0.0,0.012717,0.084393,0.115607,0.144509
3,3,39153020734,NaN,2021-12-30,apply to this job think youre the perfect cand...,False,NaN,think youre perfect candidate amazon delivery ...,"[think, youre, perfect, candidate, amazon, del...",557,...,amazon,10,0.017953,0.000000,0.007181,0.0,0.026930,0.129264,0.206463,0.220826
4,4,39153020197,3135,2021-12-29,entry level sales executive full time blooming...,False,NaN,entry level executive full time bloomington il...,"[entry, level, executive, full, time, blooming...",301,...,territory,6,0.019934,0.003322,0.000000,0.0,0.016611,0.056478,0.129568,0.152824
5,5,39153020204,3174,2021-12-29,entry level sales executive full time gainesvi...,False,NaN,entry level executive full time gainesville ga...,"[entry, level, executive, full, time, gainesvi...",328,...,territory,6,0.018293,0.003049,0.000000,0.0,0.015244,0.054878,0.125000,0.143293
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9996,9996,39153043361,27027,2021-12-30,general manager service king 31 charlotte nc s...,False,NaN,general manager service king 31 charlotte nc s...,"[general, manager, service, king, 31, charlott...",652,...,ability,10,0.015337,0.003067,0.001534,0.0,0.003067,0.075153,0.142638,0.148773
9997,9997,39153043368,NaN,2021-12-30,exercise physiologist cleveland clinic 39 lynd...,False,NaN,exercise physiologist cleveland clinic 39 lynd...,"[exercise, physiologist, cleveland, clinic, 39...",325,...,exercise,10,0.030769,0.000000,0.000000,0.0,0.049231,0.092308,0.252308,0.209231
9998,9998,39153043265,NaN,2021-12-30,server sonder social club dunedin fl quick app...,False,NaN,server sonder social club dunedin fl quick det...,"[server, sonder, social, club, dunedin, fl, qu...",132,...,cocktails,3,0.022727,0.000000,0.000000,0.0,0.000000,0.083333,0.121212,0.083333
9999,9999,39153043283,NaN,2021-12-30,restaurant server the melting pot of cooper ci...,False,NaN,restaurant server melting pot cooper city fort...,"[restaurant, server, melting, pot, cooper, cit...",187,...,restaurant,6,0.032086,0.000000,0.000000,0.0,0.000000,0.085561,0.117647,0.122995


## Prototyping 

In [ ]:
### Drop main that are not in compustat and append a year into one dataframe 
### Declare Path
path_main=Path('/Users/mgor/Documents/Strategy/2YP/Data/data_csv')
df_2021=pd.DataFrame()

### Select employers that are report on compustat

for file in path_main.glob('*.zip'):
    print(file.name)
    df_aux= pd.read_table(file, delimiter=',', header=0)
    df_aux=df_aux.rename(columns={"JobID" :"BGTJobId"})
    df_2021=df_2021.append(df_aux)
    

### Declare path
path_txt=Path('/global/home/pc_moseguera/data/Burning Glass 2/CSV/US/Add/Main/2021')

for file in path_main.glob('*.txt'):
    print(file.name)
    df_aux= pd.read_table(file, encoding='latin')
    df_2021=df_2021.merge(df_aux, how='left', on='BGTJobId')

df_2021.to_csv('/global/home/pc_moseguera/data/Burning Glass 2/merged_variables/merge_main/2021/merged_2021.zip', compression='zip', index=False)



In [ ]:
### Collapse
path_txt=Path('/Users/mgor/Documents/Strategy/2YP/Data/data_csv/test')
path_main=Path('/Users/mgor/Documents/Strategy/2YP/Data/data_csv/test/main')

for mm in range(1,13,1):
    df=pd.DataFrame()
    ### Txt by month
    for file in path_txt.glob('*.zip'):
        month_start=int(file.name[27:29])
        
        if len(file.name)<40:
            month_end=0
            
        else:
            month_end=int(file.name[36:38])
        
        
        if month_start==mm:
            df_aux= pd.read_table(file, delimiter=',', header=0)
            df_aux=df_aux[(df_aux['month']==mm)]
            df_aux=df_aux.rename(columns={"JobID" :"BGTJobId"})
            df=df.append(df_aux)
        
        elif month_end==mm:
            df_aux= pd.read_table(file, delimiter=',', header=0)
            df_aux=df_aux[(df_aux['month']==mm)]
            df_aux=df_aux.rename(columns={"JobID" :"BGTJobId"})
            df=df.append(df_aux)
    
    ### Merge with main
    for ff in path_main.glob('*.zip'):
        month=int(ff.name[10:12])
        
        if month==mm:
            df_main= pd.read_table(ff, encoding='latin')
            df=df.merge(df_main, how='left', on='BGTJobId')
    
    df.to_csv('/Users/mgor/Documents/Strategy/2YP/Data/data_csv/test/out/main'+str(mm)+'.csv.zip', compression='zip', index=False)

    ### Collapse
    df_collapsed=df.groupby(['year','month','MSA','Employer','SectorName','OccFam','OccFamName', 'SOC', 'SOCName', 'ONET', 'ONETName','NAICS6','Lat', 'Lon', 'State']).mean().reset_index()
    
    df_collapsed.to_csv('/global/home/pc_moseguera/data/Burning Glass 2/merged_variables/merge_main/2021/collapsed'+str(mm)+'.csv.zip', compression='zip', index=False)
    
    ### Occupation Family
    df_collapsed=df_collapsed.groupby(['year','month','MSA','Employer','SectorName','OccFam','OccFamName','Lat', 'Lon', 'State']).mean().reset_index()
    df_collapsed.to_csv('/global/home/pc_moseguera/data/Burning Glass 2/merged_variables/merge_main/2021/occ'+str(mm)+'.csv.zip', compression='zip', index=False)
    
    ## Company
    df_collapsed=df_collapsed.groupby(['year','month','MSA','Employer','SectorName','Lat', 'Lon', 'State']).mean().reset_index()
    df_collapsed.to_csv('/global/home/pc_moseguera/data/Burning Glass 2/merged_variables/merge_main/2021/sec'+str(mm)+'.csv.zip', compression='zip', index=False)
    
    ## MSA
    df_collapsed=df_collapsed.groupby(['year','month','MSA','Lat', 'Lon', 'State']).mean().reset_index()
    df_collapsed.to_csv('/global/home/pc_moseguera/data/Burning Glass 2/merged_variables/merge_main/2021/msa'+str(mm)+'.csv.zip', compression='zip', index=False)
    
    ### Time
    df_collapsed=df_collapsed.groupby(['year','month']).mean().reset_index()
    df_collapsed.to_csv('/global/home/pc_moseguera/data/Burning Glass 2/merged_variables/merge_main/2021/time'+str(mm)+'.csv.zip', compression='zip', index=False)
    
    del df
    del df_collapsed

    #gc.collect()

In [201]:
df_aux= pd.read_table('/Users/mgor/Documents/Strategy/2YP/Data/data_merged/2020/year.csv.zip', delimiter=',', header=0)


In [187]:
### Difference
path_txt=Path('/Users/mgor/Documents/Strategy/2YP/Data/data_merged/2020')
df=pd.DataFrame()
df_firm=pd.DataFrame()

for file in path_txt.glob('*.zip'):
        
        if file.name[0:4]=='main':
            df_aux= pd.read_table(file, delimiter=',', header=0)
            df_aux=df_aux.drop(columns=[ 'JobDate_x', 'IsDuplicate', 'IsDuplicateOf', 'WordFreq','MostCom'])
            df=df.append(df_aux)
            del df_aux
            gc.collect()
            
df_occ=df.groupby(['year','Employer', 'SOCName']).agg(
max_csr = pd.NamedAgg(column='main_dict', aggfunc=max),
min_csr = pd.NamedAgg(column='main_dict', aggfunc=min ),
p25_csr = pd.NamedAgg(column='main_dict', aggfunc=lambda x: np.percentile (x, 25)),
p75_csr = pd.NamedAgg(column='main_dict', aggfunc=lambda x: np.percentile (x, 75)),
p905_csr = pd.NamedAgg(column='main_dict', aggfunc=lambda x: np.percentile (x, 90)),
p10_csr = pd.NamedAgg(column='main_dict', aggfunc=lambda x: np.percentile (x, 10)),
count_csr = pd.NamedAgg(column='main_dict', aggfunc=lambda x: x.count())).reset_index()

df_occ.to_csv('/Users/mgor/Documents/Strategy/2YP/Data/data_csv/test/out/difference_occ.csv.zip', compression='zip', index=False)

del df_occ
gc.collect()

df_firm=df.groupby(['year','Employer']).agg(
max_csr = pd.NamedAgg(column='main_dict', aggfunc=max),
min_csr = pd.NamedAgg(column='main_dict', aggfunc=min ),
p25_csr = pd.NamedAgg(column='main_dict', aggfunc=lambda x: np.percentile (x, 25)),
p75_csr = pd.NamedAgg(column='main_dict', aggfunc=lambda x: np.percentile (x, 75)),
p90_csr = pd.NamedAgg(column='main_dict', aggfunc=lambda x: np.percentile (x, 90)),
p10_csr = pd.NamedAgg(column='main_dict', aggfunc=lambda x: np.percentile (x, 10)),
count_csr = pd.NamedAgg(column='main_dict', aggfunc=lambda x: x.count())).reset_index()

df_firm.to_csv('/Users/mgor/Documents/Strategy/2YP/Data/data_csv/test/out/difference_firm.csv.zip', compression='zip', index=False)

del df_firm
gc.collect()
                             

/opt/anaconda3/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3063: DtypeWarning: Columns (22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,46,47,48,49,50,51,54,55,56,58,61,62,69,70,71,72) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


0

In [194]:
### Graph Time Series
df=pd.DataFrame()

for year in range(2010,2022,1):
    path_month=Path('/global/home/pc_moseguera/data/Burning Glass 2/merged_variables/merge_main/'+str(year))

    for file in path_txt.glob('*.zip'):

            if file.name[0:4]=='time':
                df_aux= pd.read_table(file, delimiter=',', header=0)
                df_aux=df_aux.drop(columns=[ 'Unnamed: 0', 'IsDuplicate', 'IsDuplicateOf'])
                df=df.append(df_aux)
                del df_aux
                gc.collect()

df.to_csv('/global/home/pc_moseguera/data/Burning Glass 2/merged_variables/merge_main/year.csv.zip', compression='zip', index=False)



In [ ]:
### Graph Differences 
df=pd.DataFrame()

for year in range(2010,2022,1):
    path_month=Path('/global/home/pc_moseguera/data/Burning Glass 2/merged_variables/merge_main/'+str(year))

    for file in path_txt.glob('*.zip'):

            if file.name[0:4]=='time':
                df_aux= pd.read_table(file, delimiter=',', header=0)
                df=df.append(df_aux)
                del df_aux
                gc.collect()

df.to_csv('/global/home/pc_moseguera/data/Burning Glass 2/merged_variables/merge_main/year.csv.zip', compression='zip', index=False)




In [204]:
### Graph Differences 
df=pd.DataFrame()

for year in range(2010,2022,1):
    path_month=Path('/global/home/pc_moseguera/data/Burning Glass 2/merged_variables/merge_main/'+str(year))

    for file in path_txt.glob('*.zip'):

            if file.name[11:14]=='occ':
                df_aux= pd.read_table(file, delimiter=',', header=0)
                df=df.append(df_aux)
                del df_aux
                gc.collect()

df.to_csv('/global/home/pc_moseguera/data/Burning Glass 2/merged_variables/merge_main/diff_occ.csv.zip', compression='zip', index=False)




p


In [ ]:
### Required Degree 

In [255]:
df_x= pd.read_table('/Users/mgor/Documents/Strategy/2YP/Data/data_csv/Skills BGT/Degree_2022-01.zip',  encoding='latin')

In [256]:
df_x=df_x.drop(columns=[ 'JobDate', 'Salary'])

In [257]:
df_x=df_x.groupby(['BGTJobId']).min()

In [249]:
len(df_x.BGTJobId.unique())

2255980

In [258]:
len(df_x)

2255980

In [ ]:
### Skills



month=['01','02','03','04','05','06','07','08','09','10','11','12']
for mm in month:
    ### Read main
    df= pd.read_table('/global/home/pc_moseguera/data/Burning Glass 2/merged_variables/merge_main/2018/main'+mm+'.csv.zip', delimiter=',', header=0)
    df=df.drop(columns=[ 'JobDate_x', 'IsDuplicate', 'IsDuplicateOf', 'WordFreq','MostCom'])
    
    ### Skills by month
    
    df_skills= pd.read_table('/global/home/pc_moseguera/data/Burning Glass 2/CSV/US/Add/Skill/2018/Skills_2018-'+mm+'.zip', encoding='latin')
    df_skills=df_skills.drop(columns=[ 'JobDate', 'Skill', 'SkillCluster', 'SkillClusterFamily',
        'IsBaseline', 'IsSoftware', 'Salary'])
    df_skills=df_skills.groupby(['BGTJobId']).max().reset_index()
    df=df.append(df_skills, how='left', on='BGTJobId')
    
    
    ### Degree by month
 
    df_degree= pd.read_table('/global/home/pc_moseguera/data/Burning Glass 2/CSV/US/Add/Degree/2018/Degree_2018-'+mm+'.zip', encoding='latin')
    df_degree=df_degree.drop(columns=[ 'JobDate', 'Salary'])
    df_degree=df_degree.groupby(['BGTJobId']).min().reset_index()
    df=df.merge(df_degree, how='left', on='BGTJobId')
    
    df.to_csv('/Users/mgor/Documents/Strategy/2YP/Data/data_csv/test/out/mainfull'+mm+'.csv.zip', compression='zip', index=False)

    ### Collapse
    
    df_collapsed=df.groupby(['year','month','Employer']).mean().reset_index()

    del df
    del df_collapsed

    gc.collect()

In [259]:
l=['01','02','03','04','05','06','07','08','09','10','11','12']

In [264]:
for mm in l:
    print(str(int(mm)))

1
2
3
4
5
6
7
8
9
10


In [297]:
df_xx= pd.read_table('/Users/mgor/Documents/Strategy/2YP/Data/data_merged/Merged_main/spec.csv.zip', delimiter=',', header=0)
df_xx=df_xx[(df_xx.main_dict < 1) ]

In [272]:
df_xx.IsSpecialized.unique()

array([nan,  1.,  0.])

In [277]:
len(df_xx[df_xx.IsSpecialized == 'nan'])

0

In [282]:
df_xx.IsSpecialized.describe()

count    10229.000000
mean         0.987005
std          0.087284
min          0.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          1.000000
Name: IsSpecialized, dtype: float64

In [298]:
df_xx

,year,Employer,month,BGTJobId,length,Freq,Rel_Freq,Full_wilzha,career_meaning,worklife_meaning,...,MaxEdu,Exp,MaxExp,MinSalary,MaxSalary,MinHrlySalary,MaxHrlySalary,Internship,IsSpecialized,DegreeLevel
0,2010,1-800-Flowers.com,7.000000,2.989088e+08,279.162363,10.356277,0.039710,0.000427,0.000192,0.000000,...,-700.339461,-339.645596,-606.197602,-999.000000,-999.000000,-999.000000,-999.000000,0.000000,1.000000,15.695476
1,2010,1901 Group,8.000000,2.968980e+08,215.000000,5.000000,0.023256,0.000000,0.000000,0.000000,...,16.000000,-999.000000,3.000000,20000.000000,24000.000000,9.620000,11.540000,0.000000,1.000000,13.000000
2,2010,20/20 Companies,6.500000,3.007111e+08,260.590152,10.415437,0.044289,0.000556,0.000928,0.000000,...,-986.047321,-827.781035,-734.696862,26494.342014,39005.057887,-249.258068,-243.243209,0.000000,1.000000,12.404762
3,2010,2020 Companies,6.875000,2.989468e+08,272.760417,7.447917,0.029763,0.002812,0.001832,0.000000,...,-977.750000,-665.604167,-915.583333,25037.937500,31046.270833,-424.815417,-421.926458,0.000000,1.000000,12.000000
4,2010,2020 Company,7.800000,3.011630e+08,84.166667,4.400000,0.051810,0.000000,0.000000,0.000000,...,-999.000000,-999.000000,-999.000000,17575.700000,19655.700000,-490.810000,-489.810000,0.000000,1.000000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1036608,2021,inVentiv Health,6.500000,3.903563e+10,316.982639,8.189583,0.025562,0.000807,0.004068,0.000000,...,4.043750,-156.050694,-905.034722,-999.000000,-999.000000,-999.000000,-999.000000,0.000000,1.000000,16.250000
1036609,2021,"lowes foods, llc",9.333333,3.909017e+10,189.425750,6.037892,0.032029,0.000319,0.000000,0.000000,...,0.000000,-989.199367,-989.189583,542.495575,793.911746,-943.117953,-942.997001,0.000000,1.000000,13.333333
1036610,2021,na,7.250000,3.904993e+10,305.976142,8.560814,0.035060,0.001988,0.000347,0.000111,...,1.496945,-596.048745,-959.890675,19198.185005,22405.778543,-570.910760,-569.412327,0.009469,0.949329,14.125057
1036611,2021,rpc cass regional missouri,12.000000,3.915374e+10,196.000000,7.000000,0.035714,0.000000,0.000000,0.000000,...,0.000000,5.000000,-999.000000,-999.000000,-999.000000,-999.000000,-999.000000,0.000000,1.000000,16.000000


In [299]:
len(df_xx[df_xx.main_dict>1])

0

In [315]:
df_xxx.main_dict.describe()

count    625040.000000
mean          0.066412
std           0.034026
min           0.000000
25%           0.045455
50%           0.061677
75%           0.083900
max           1.000000
Name: main_dict, dtype: float64

In [561]:
df_xx= pd.read_table('/Users/mgor/Documents/Strategy/2YP/Data/data_csv/test/no_text_US_XML_AddFeed_20121223_20121229.csv.zip', delimiter=',', header=0)



In [652]:
df_xx.columns

Index(['Unnamed: 0', 'JobID', 'CanonEmployer', 'JobReferenceID', 'JobDate',
       'IsDuplicate', 'IsDuplicateOf', 'length', 'WordFreq', 'MostCom',
       'ComWord', 'Freq', 'Rel_Freq', 'Full_wilzha', 'career_meaning',
       'worklife_meaning', 'SDGs', 'CSR', 'main_dict', 'employee',
       'environment', 'human_rights', 'soc_com', 'year', 'month'],
      dtype='object')

In [565]:
df_test= pd.read_table('/Users/mgor/Documents/Strategy/2YP/Data/data_merged/Merged_main/main1.csv.zip', delimiter=',', header=0)



/opt/anaconda3/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3063: DtypeWarning: Columns (50,51) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [567]:
df_test.columns

Index(['Unnamed: 0', 'BGTJobId', 'CanonEmployer', 'JobReferenceID',
       'JobDate_x', 'IsDuplicate', 'IsDuplicateOf', 'length', 'WordFreq',
       'MostCom', 'ComWord', 'Freq', 'Rel_Freq', 'Full_wilzha',
       'career_meaning', 'worklife_meaning', 'SDGs', 'CSR', 'Recru_sig',
       'main_dict', 'year', 'month', 'JobId', 'JobDate_y', 'CleanTitle',
       'CanonTitle', 'OccFam', 'OccFamName', 'SOC', 'SOCName', 'ONET',
       'ONETName', 'Specialty', 'BGTOcc', 'BGTOccName', 'BGTOccGroupName',
       'BGTOccGroupName2', 'BGTCareerAreaName', 'BGTCareerAreaName2',
       'Employer', 'Sector', 'SectorName', 'NAICS3', 'NAICS4', 'NAICS5',
       'NAICS6', 'City', 'State', 'County', 'FIPSState', 'FIPSCounty', 'FIPS',
       'Lat', 'Lon', 'BestFitMSA', 'BestFitMSAName', 'BestFitMSAType', 'MSA',
       'MSAName', 'Edu', 'MaxEdu', 'Degree', 'MaxDegree', 'Exp', 'MaxExp',
       'MinSalary', 'MaxSalary', 'MinHrlySalary', 'MaxHrlySalary',
       'PayFrequency', 'SalaryType', 'JobHours', 'TaxTerm', 

In [568]:
df_msa=df_test.groupby(['year','MSA']).agg(
                max_csr = pd.NamedAgg(column='main_dict', aggfunc=max),
                min_csr = pd.NamedAgg(column='main_dict', aggfunc=min ),
                p25_csr = pd.NamedAgg(column='main_dict', aggfunc=lambda x: np.percentile (x, 25)),
                p75_csr = pd.NamedAgg(column='main_dict', aggfunc=lambda x: np.percentile (x, 75)),
                p905_csr = pd.NamedAgg(column='main_dict', aggfunc=lambda x: np.percentile (x, 90)),
                p10_csr = pd.NamedAgg(column='main_dict', aggfunc=lambda x: np.percentile (x, 10)),
                count_csr = pd.NamedAgg(column='main_dict', aggfunc=)).reset_index()

In [637]:
def max_min(x):
    return x.max() - x.min()

def per_90(x):
    return x.quantile(0.90)

def per_10(x):
    return x.quantile(0.10)

In [644]:
## MSA Aggregate
df_msa=df.groupby(['year','MSA','State']).agg(({'main_dict':['max','min',per_90, per_10,'median','count','mean','std',max_min],
                                            'employee':['max','min','median','mean','std', max_min],
                                            'environment':['max','min','median','mean','std',max_min],
                                            'human_rights':['max','min','median','mean','std',max_min],
                                            'soc_com':['max','min','median','mean','std', max_min],})).reset_index()
df_msa.columns=['_'.join(col) if type(col) is tuple else col for col in df_msa.columns.values]
df_msa.to_csv('/global/home/pc_moseguera/data/Burning Glass 2/merged_variables/merge_main/'+str(yy)+'/msa'+str(mm)+'.csv.zip', compression='zip', index=False)
del df_msa

In [645]:
## Main aggregate
df_collapsed=df.groupby(['year','month','MSA','Employer','SectorName','OccFam','OccFamName', 'SOC', 'SOCName', 'ONET', 'ONETName','NAICS6','Lat', 'Lon', 'State']).agg(({'main_dict':['max','min',per_90, per_10,'median','count','mean','std',max_min],
                                            'employee':['max','min','median','mean','std', max_min],
                                            'environment':['max','min','median','mean','std',max_min],
                                            'human_rights':['max','min','median','mean','std',max_min],
                                            'soc_com':['max','min','median','mean','std', max_min],})).reset_index()
df_collapsed.columns=['_'.join(col) if type(col) is tuple else col for col in df_collapsed.columns.values]
df_collapsed.to_csv('/global/home/pc_moseguera/data/Burning Glass 2/merged_variables/merge_main/'+str(yy)+'/collapsed'+str(mm)+'.csv.zip', compression='zip', index=False)
del df_collapsed

In [ ]:
## Sector-Time

df_sec=df.groupby(['year','SectorName']).agg(({'main_dict':['max','min','median','count','mean','std',max_min],
                                            'employee':['max','min','median','mean','std', max_min],
                                            'environment':['max','min','median','mean','std',max_min],
                                            'human_rights':['max','min','median','mean','std',max_min],
                                            'soc_com':['max','min','median','mean','std', max_min],})).reset_index()
df_sec.columns=['_'.join(col) if type(col) is tuple else col for col in df_sec.columns.values]
df_sec.to_csv('/global/home/pc_moseguera/data/Burning Glass 2/merged_variables/merge_main/'+str(yy)+'/sec'+str(mm)+'.csv.zip', compression='zip', index=False)
del df_sec


In [ ]:
## Occ-Time
df_occ=df.groupby(['year','OccFamName']).agg(({'main_dict':['max','min','median','count','mean','std',max_min],
                                            'employee':['max','min','median','mean','std', max_min],
                                            'environment':['max','min','median','mean','std',max_min],
                                            'human_rights':['max','min','median','mean','std',max_min],
                                            'soc_com':['max','min','median','mean','std', max_min],})).reset_index()
df_occ.columns=['_'.join(col) if type(col) is tuple else col for col in df_occ.columns.values]

df_occ.to_csv('/global/home/pc_moseguera/data/Burning Glass 2/merged_variables/merge_main/'+str(yy)+'/occ'+str(mm)+'.csv.zip', compression='zip', index=False)

del df_occ

In [647]:
### Differences across MSA within the same firm within the same occupation

df_fim_occ=df.groupby(['year','Employer','OccFamName']).agg(({'main_dict':['max','min','median','count','mean','std',max_min],
                                            'employee':['max','min','median','mean','std', max_min],
                                            'environment':['max','min','median','mean','std',max_min],
                                            'human_rights':['max','min','median','mean','std',max_min],
                                            'soc_com':['max','min','median','mean','std', max_min]})).reset_index()
df_fim_occ.columns=['_'.join(col) if type(col) is tuple else col for col in df_fim_occ.columns.values]

df_fim_occ.to_csv('/global/home/pc_moseguera/data/Burning Glass 2/merged_variables/merge_main/'+str(yy)+'/fim_occ'+str(mm)+'.csv.zip', compression='zip', index=False)
del fim_occ

In [ ]:
## Difference in CSR within firms across occupations and MSAs
df_fim=df.groupby(['year','Employer']).agg(({'main_dict':['max','min','median','count','mean','std',max_min],
                                            'employee':['max','min','median','mean','std', max_min],
                                            'environment':['max','min','median','mean','std',max_min],
                                            'human_rights':['max','min','median','mean','std',max_min],
                                            'soc_com':['max','min','median','mean','std', max_min],})).reset_index()
df_fim.columns=['_'.join(col) if type(col) is tuple else col for col in df_fim.columns.values]
df_fim.to_csv('/global/home/pc_moseguera/data/Burning Glass 2/merged_variables/merge_main/'+str(yy)+'/fim'+str(mm)+'.csv.zip', compression='zip', index=False)


In [ ]:
### Time
df_time=df.groupby(['year']).agg(({'main_dict':['max','min',per_90, per_10,'median','count','mean','std',max_min],
                                            'employee':['max','min','median','mean','std', max_min],
                                            'environment':['max','min','median','mean','std',max_min],
                                            'human_rights':['max','min','median','mean','std',max_min],
                                            'soc_com':['max','min','median','mean','std', max_min],})).reset_index()
df_time.columns=['_'.join(col) if type(col) is tuple else col for col in df_time.columns.values]

df_time.to_csv('/global/home/pc_moseguera/data/Burning Glass 2/merged_variables/merge_main/'+str(yy)+'/time'+str(mm)+'.csv.zip', compression='zip', index=False)




In [650]:
len(df_test)

625045

In [ ]:
### Collapse
    df_collapsed=df.groupby(['year','month','MSA','Employer','SectorName','OccFam','OccFamName', 'SOC', 'SOCName', 'ONET', 'ONETName','NAICS6','Lat', 'Lon', 'State']).mean().reset_index()
    
    df_collapsed.to_csv('/global/home/pc_moseguera/data/Burning Glass 2/merged_variables/merge_main/2019/collapsed'+str(mm)+'.csv.zip', compression='zip', index=False)
    
    ### Occupation Family
    df_collapsed=df_collapsed.groupby(['year','month','MSA','Employer','SectorName','OccFam','OccFamName','Lat', 'Lon', 'State']).mean().reset_index()
    df_collapsed.to_csv('/global/home/pc_moseguera/data/Burning Glass 2/merged_variables/merge_main/2019/occ'+str(mm)+'.csv.zip', compression='zip', index=False)
    
    ## Company
    df_collapsed=df_collapsed.groupby(['year','month','MSA','Employer','SectorName','Lat', 'Lon', 'State']).mean().reset_index()
    df_collapsed.to_csv('/global/home/pc_moseguera/data/Burning Glass 2/merged_variables/merge_main/2019/sec'+str(mm)+'.csv.zip', compression='zip', index=False)
    
    ## MSAs
    df_collapsed=df_collapsed.groupby(['year','month','MSA','Lat', 'Lon', 'State']).mean().reset_index()
    df_collapsed.to_csv('/global/home/pc_moseguera/data/Burning Glass 2/merged_variables/merge_main/2019/msa'+str(mm)+'.csv.zip', compression='zip', index=False)
    
    ### Time
    df_collapsed=df_collapsed.groupby(['year','month']).mean().reset_index()
    df_collapsed.to_csv('/global/home/pc_moseguera/data/Burning Glass 2/merged_variables/merge_main/2019/time'+str(mm)+'.csv.zip', compression='zip', index=False)
    